# Notebook 5: MPC for Real-Time Balancing and Imbalance Management

## References

1. **Oldewurtel, F., Parisio, A., Jones, C. N., et al. (2012).** *"Use of model predictive control and weather forecasts for energy efficient building climate control."* Applied Energy, 93, 15–26. [DOI: 10.1016/j.apenergy.2011.12.094](https://doi.org/10.1016/j.apenergy.2011.12.094)

2. **Parisio, A., Rikos, E., Glielmo, L. (2014).** *"A model predictive control approach to microgrid operation optimization."* IEEE Transactions on Control Systems Technology, 22(5), 1813–1827. [DOI: 10.1109/TCST.2013.2295737](https://doi.org/10.1109/TCST.2013.2295737)

3. **Pérez, E., Beltran, H., Aparicio, N., Rodríguez, P. (2013).** *"Predictive power control for PV plants with energy storage."* IEEE Transactions on Sustainable Energy, 4(2), 482–490. [DOI: 10.1109/TSTE.2012.2210255](https://doi.org/10.1109/TSTE.2012.2210255)

## What is implemented below

We simulate a **full-day MPC loop** for a BRP portfolio:
1. Day-ahead: BRP submits a nomination
2. Real-time: every 15 min, MPC re-optimizes BESS dispatch to **track the nomination**
3. Imbalance = deviation between actual portfolio position and nomination
4. Comparison: MPC tracking vs "fire-and-forget" (execute DA plan blindly)

Key insight: MPC minimizes **imbalance cost only** (DA cost is already committed).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pulp

np.random.seed(42)


## 1. Setup


In [ ]:
T_da = 24; T_rt = 96; dt_da = 1.0; dt_rt = 0.25
N = 3; eta_ch = 0.95; eta_dis = 0.95

site_params = [(6.0, 2.0, 10.0, 5.0), (8.0, 3.0, 12.0, 6.0), (4.0, 1.5, 7.0, 3.5)]

hours_da = np.arange(T_da)
hours_rt = np.arange(T_rt) * dt_rt

# DA forecasts (hourly)
pv_fcst_da, load_fcst_da = [], []
for i, (pp, lb, _, _) in enumerate(site_params):
    pv_fcst_da.append(np.maximum(0, pp * np.exp(-0.5*((hours_da-12)/3)**2)))
    load_fcst_da.append(lb + 0.7*np.exp(-0.5*((hours_da-7.5)/2)**2) + 1.0*np.exp(-0.5*((hours_da-19)/2.5)**2))

# DA prices
price_da = np.maximum(40+20*np.sin(2*np.pi*(hours_da-6)/24)+10*np.exp(-0.5*((hours_da-18)/3)**2), 10.0)/1000
price_imb_def = price_da * 1.4   # deficit premium (40% above DA)
price_imb_sur = price_da * 0.6   # surplus return (40% below DA)

# Real-time actuals (15-min, with cloud events and load noise)
pv_actual, load_actual = [], []
for i, (pp, lb, _, _) in enumerate(site_params):
    pv_base = np.maximum(0, pp * np.exp(-0.5*((hours_rt-12)/3)**2))
    clouds = np.ones(T_rt)
    for _ in range(4):
        tc = np.random.randint(24, 72)
        dur = np.random.randint(3, 10)
        clouds[tc:tc+dur] *= 0.2 + 0.3*np.random.rand()
    pv_actual.append(np.maximum(0, pv_base * clouds + 0.1*np.random.randn(T_rt)))
    load_base = lb + 0.7*np.exp(-0.5*((hours_rt-7.5)/2)**2) + 1.0*np.exp(-0.5*((hours_rt-19)/2.5)**2)
    load_actual.append(np.maximum(0.3, load_base + 0.4*np.random.randn(T_rt)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i in range(N):
    axes[i].plot(hours_rt, pv_actual[i], 'gold', alpha=0.7, label='PV actual')
    axes[i].plot(hours_da, pv_fcst_da[i], 'orange', lw=2, ls='--', label='PV DA')
    axes[i].plot(hours_rt, load_actual[i], 'red', alpha=0.7, label='Load actual')
    axes[i].plot(hours_da, load_fcst_da[i], 'darkred', lw=2, ls='--', label='Load DA')
    axes[i].set_title(f'Site {i}'); axes[i].set_xlabel('Hour')
    axes[i].legend(fontsize=6); axes[i].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/tmp/nb5_profiles.png', dpi=100); plt.show()


## 2. Day-Ahead Nomination


In [ ]:
prob_da = pulp.LpProblem("DA", pulp.LpMinimize)
pc_da = {}; pd_da = {}; sc_da = {}
for i in range(N):
    Em, Pm = site_params[i][2], site_params[i][3]
    for t in range(T_da):
        pc_da[i,t] = pulp.LpVariable(f"c{i}{t}", 0, Pm)
        pd_da[i,t] = pulp.LpVariable(f"d{i}{t}", 0, Pm)
        sc_da[i,t] = pulp.LpVariable(f"e{i}{t}", Em*0.1, Em)
pb = [pulp.LpVariable(f"pb{t}", 0) for t in range(T_da)]
ps = [pulp.LpVariable(f"ps{t}", 0) for t in range(T_da)]
prob_da += pulp.lpSum([price_da[t]*(pb[t]-ps[t])*dt_da for t in range(T_da)])
for t in range(T_da):
    for i in range(N):
        Em = site_params[i][2]; sp = Em*0.5 if t==0 else sc_da[i,t-1]
        prob_da += sc_da[i,t] == sp + eta_ch*pc_da[i,t]*dt_da - pd_da[i,t]*dt_da/eta_dis
    prob_da += (pulp.lpSum([load_fcst_da[i][t]+pc_da[i,t]-pv_fcst_da[i][t]-pd_da[i,t]
                for i in range(N)]) == pb[t]-ps[t])
prob_da.solve(pulp.PULP_CBC_CMD(msg=0))

nomination = np.array([pulp.value(pb[t]) - pulp.value(ps[t]) for t in range(T_da)])
nom_rt = np.repeat(nomination, 4)  # expand to 15-min
da_plan_ch = {i: np.array([pulp.value(pc_da[i,t]) for t in range(T_da)]) for i in range(N)}
da_plan_dis = {i: np.array([pulp.value(pd_da[i,t]) for t in range(T_da)]) for i in range(N)}

print(f"DA nomination computed. Total energy: {np.sum(nomination):.1f} kWh")


## 3. MPC Loop

The MPC objective (given fixed nomination) is to minimize **imbalance cost only**:

$$\min \sum_{t} \left[ \pi^{def}_t \cdot \delta^-_t - \pi^{sur}_t \cdot \delta^+_t \right]$$

where $\delta^+_t = \max(N_t - A_t, 0)$ (surplus) and $\delta^-_t = \max(A_t - N_t, 0)$ (deficit).

Since $\pi^{def} > \pi^{sur}$, the MPC drives the actual position $A_t$ toward the nomination $N_t$.


In [ ]:
mpc_call_counter = [0]

def solve_mpc(soc_now, pv_fcast, load_fcast, nom_remaining, T_h, dt, params,
              price_def, price_sur):
    """MPC: minimize imbalance cost by adjusting BESS dispatch to track nomination."""
    N_s = len(params)
    T_h = min(T_h, len(nom_remaining), min(len(pv_fcast[i]) for i in range(N_s)))
    if T_h <= 0:
        return {i: (0, 0) for i in range(N_s)}
    
    mpc_call_counter[0] += 1
    pfx = f"m{mpc_call_counter[0]}_"
    prob = pulp.LpProblem(f"MPC_{mpc_call_counter[0]}", pulp.LpMinimize)
    pc = {}; pd = {}; sc = {}
    for i in range(N_s):
        Em, Pm = params[i][2], params[i][3]
        for t in range(T_h):
            pc[i,t] = pulp.LpVariable(f"{pfx}c{i}_{t}", 0, Pm)
            pd[i,t] = pulp.LpVariable(f"{pfx}di{i}_{t}", 0, Pm)
            sc[i,t] = pulp.LpVariable(f"{pfx}e{i}_{t}", Em*0.1, Em)
    
    actual = [pulp.LpVariable(f"{pfx}a{t}") for t in range(T_h)]
    sur = [pulp.LpVariable(f"{pfx}s{t}", 0) for t in range(T_h)]
    dfc = [pulp.LpVariable(f"{pfx}df{t}", 0) for t in range(T_h)]
    
    # Objective: minimize imbalance cost only
    prob += pulp.lpSum([
        price_def[min(t,len(price_def)-1)] * dfc[t] * dt -
        price_sur[min(t,len(price_sur)-1)] * sur[t] * dt
        for t in range(T_h)])
    
    for t in range(T_h):
        for i in range(N_s):
            Em = params[i][2]
            sp = soc_now[i] if t==0 else sc[i,t-1]
            prob += sc[i,t] == sp + eta_ch*pc[i,t]*dt - pd[i,t]*dt/eta_dis
        
        prob += (pulp.lpSum([load_fcast[i][t]+pc[i,t]-pv_fcast[i][t]-pd[i,t]
                 for i in range(N_s)]) == actual[t])
        
        nom_t = nom_remaining[min(t, len(nom_remaining)-1)]
        prob += nom_t - actual[t] == sur[t] - dfc[t]
    
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    if prob.status != 1:
        return {i: (0, 0) for i in range(N_s)}
    return {i: (pulp.value(pc[i,0]), pulp.value(pd[i,0])) for i in range(N_s)}

# MPC simulation
MPC_HORIZON = 16
soc_traj = {i: [site_params[i][2]*0.5] for i in range(N)}
actual_port = np.zeros(T_rt)

# Expand hourly prices to 15-min for settlement
price_def_rt = np.repeat(price_imb_def, 4)
price_sur_rt = np.repeat(price_imb_sur, 4)

print("Running MPC simulation...")
for t in range(T_rt):
    soc_now = {i: soc_traj[i][-1] for i in range(N)}
    remaining = T_rt - t
    h = t // 4
    
    # Updated forecast: blend actual (near) and DA forecast (far)
    pv_f, load_f = [], []
    for i in range(N):
        flen = min(MPC_HORIZON, remaining)
        pf = np.zeros(flen); lf = np.zeros(flen)
        for k in range(flen):
            tf = t + k
            if tf < T_rt:
                blend = min(1.0, k / 8.0)  # 0 = actual, 1 = DA forecast
                pf[k] = (1-blend)*pv_actual[i][tf] + blend*pv_fcst_da[i][min(tf//4, T_da-1)]
                lf[k] = (1-blend)*load_actual[i][tf] + blend*load_fcst_da[i][min(tf//4, T_da-1)]
                pf[k] = max(pf[k], 0); lf[k] = max(lf[k], 0.3)
        pv_f.append(pf); load_f.append(lf)
    
    actions = solve_mpc(soc_now, pv_f, load_f, nom_rt[t:t+MPC_HORIZON],
                         MPC_HORIZON, dt_rt, site_params,
                         price_imb_def[h:h+MPC_HORIZON], price_imb_sur[h:h+MPC_HORIZON])
    
    net = 0
    for i in range(N):
        pci, pdi = actions[i]
        new_soc = soc_now[i] + eta_ch*pci*dt_rt - pdi*dt_rt/eta_dis
        new_soc = np.clip(new_soc, site_params[i][2]*0.1, site_params[i][2])
        soc_traj[i].append(new_soc)
        net += load_actual[i][t] + pci - pv_actual[i][t] - pdi
    actual_port[t] = net
    
    if t % 24 == 0:
        print(f"  Step {t}/{T_rt} (h={t*dt_rt:.1f})")

print("Done.")


## 4. Results


In [ ]:
# MPC imbalance
imb_mpc = nom_rt - actual_port
sur_mpc = np.maximum(imb_mpc, 0)
def_mpc = np.maximum(-imb_mpc, 0)

# Fire-and-forget: apply DA plan with actual PV/load
ff_port = np.zeros(T_rt)
ff_soc = {i: [site_params[i][2]*0.5] for i in range(N)}
for t in range(T_rt):
    h = min(t//4, T_da-1)
    for i in range(N):
        pci = da_plan_ch[i][h]; pdi = da_plan_dis[i][h]
        # Check SoC bounds
        s = ff_soc[i][-1]
        s_new = s + eta_ch*pci*dt_rt - pdi*dt_rt/eta_dis
        if s_new > site_params[i][2]: pci = 0  # can't charge more
        if s_new < site_params[i][2]*0.1: pdi = 0  # can't discharge more
        s_new = s + eta_ch*pci*dt_rt - pdi*dt_rt/eta_dis
        s_new = np.clip(s_new, site_params[i][2]*0.1, site_params[i][2])
        ff_soc[i].append(s_new)
        ff_port[t] += load_actual[i][t] + pci - pv_actual[i][t] - pdi

imb_ff = nom_rt - ff_port
sur_ff = np.maximum(imb_ff, 0)
def_ff = np.maximum(-imb_ff, 0)

# Costs
da_cost = np.sum(np.repeat(price_da, 4) * nom_rt * dt_rt)
imb_cost_mpc = np.sum(price_def_rt * def_mpc * dt_rt - price_sur_rt * sur_mpc * dt_rt)
imb_cost_ff = np.sum(price_def_rt * def_ff * dt_rt - price_sur_rt * sur_ff * dt_rt)

print(f"{'='*55}")
print(f"{'Metric':<35} {'MPC':>8} {'No-MPC':>8}")
print(f"{'-'*55}")
print(f"{'DA cost (fixed, EUR):':<35} {da_cost:8.3f} {da_cost:8.3f}")
print(f"{'Imbalance cost (EUR):':<35} {imb_cost_mpc:8.3f} {imb_cost_ff:8.3f}")
print(f"{'Total settlement (EUR):':<35} {da_cost+imb_cost_mpc:8.3f} {da_cost+imb_cost_ff:8.3f}")
print(f"{'Imbalance volume (kWh):':<35} {np.sum(np.abs(imb_mpc))*dt_rt:8.1f} {np.sum(np.abs(imb_ff))*dt_rt:8.1f}")
print(f"{'Max |imbalance| (kW):':<35} {np.max(np.abs(imb_mpc)):8.2f} {np.max(np.abs(imb_ff)):8.2f}")
print(f"{'RMSE vs nomination (kW):':<35} {np.sqrt(np.mean(imb_mpc**2)):8.2f} {np.sqrt(np.mean(imb_ff**2)):8.2f}")
print(f"{'='*55}")
saving = imb_cost_ff - imb_cost_mpc
print(f"MPC imbalance cost saving: {saving:.3f} EUR ({100*saving/max(abs(imb_cost_ff),0.01):.1f}%)")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].step(hours_rt, nom_rt, 'b-', where='mid', lw=2, label='Nomination')
axes[0,0].step(hours_rt, actual_port, 'r-', where='mid', lw=1, alpha=0.8, label='MPC actual')
axes[0,0].step(hours_rt, ff_port, 'gray', where='mid', lw=1, alpha=0.5, label='No-MPC actual')
axes[0,0].set_xlabel('Hour'); axes[0,0].set_ylabel('kW')
axes[0,0].set_title('Nomination vs Actual'); axes[0,0].legend(fontsize=7); axes[0,0].grid(alpha=0.3)

axes[0,1].fill_between(hours_rt, 0, imb_mpc, where=imb_mpc>0, color='teal', alpha=0.4, label='MPC surplus')
axes[0,1].fill_between(hours_rt, 0, imb_mpc, where=imb_mpc<0, color='coral', alpha=0.4, label='MPC deficit')
axes[0,1].step(hours_rt, imb_ff, 'gray', where='mid', lw=1.5, alpha=0.6, label='No-MPC')
axes[0,1].set_xlabel('Hour'); axes[0,1].set_ylabel('kW')
axes[0,1].set_title('Imbalance'); axes[0,1].legend(fontsize=7); axes[0,1].grid(alpha=0.3)

for i in range(N):
    axes[1,0].plot(hours_rt, soc_traj[i][:-1], '-', lw=1.5, label=f'Site {i} MPC')
    axes[1,0].plot(hours_rt, ff_soc[i][:-1], '--', lw=1, alpha=0.4)
axes[1,0].set_xlabel('Hour'); axes[1,0].set_ylabel('kWh')
axes[1,0].set_title('SoC: MPC (solid) vs No-MPC (dashed)'); axes[1,0].legend(fontsize=7)
axes[1,0].grid(alpha=0.3)

cum_mpc = np.cumsum((price_def_rt*def_mpc - price_sur_rt*sur_mpc)*dt_rt)
cum_ff = np.cumsum((price_def_rt*def_ff - price_sur_rt*sur_ff)*dt_rt)
axes[1,1].plot(hours_rt, cum_mpc, 'b-', lw=2, label='MPC')
axes[1,1].plot(hours_rt, cum_ff, 'r--', lw=2, label='No-MPC')
axes[1,1].set_xlabel('Hour'); axes[1,1].set_ylabel('EUR')
axes[1,1].set_title('Cumulative Imbalance Cost'); axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('/tmp/nb5_results.png', dpi=100); plt.show()


## 5. Key Insights for the Thesis

1. **MPC reduces imbalance cost** by re-optimizing battery dispatch in real-time as actual PV/load deviate from forecasts. The batteries absorb forecast errors, steering the portfolio toward the nomination.

2. **Rolling horizon + updated forecasts**: near-term forecasts (0-30 min) are nearly perfect, allowing precise imbalance correction. Longer-horizon MPC provides anticipation (e.g., pre-charging before expected evening peak).

3. **Integration with company's existing MPC**: the current per-site 1-minute MPC can be extended to portfolio-level nomination tracking. The key change: each site's MPC receives a **target net exchange** (from the portfolio optimizer) instead of just minimizing its own cost.

4. **Settlement timing matters**: CZ (OTE) settles imbalances at 15-minute intervals. The MPC should target minimizing 15-minute average imbalance, not instantaneous power.

5. **Combining with Notebooks 2-3**: use centralized MILP or stochastic programming for day-ahead nomination, then MPC for real-time tracking. This two-level approach is the recommended architecture.

6. **MPC horizon**: 4 hours (16 × 15-min steps) provides good anticipation without excessive computation. Longer horizons help for evening peak preparation.

---

*Next: Notebook 6 addresses the BRP–Prosumer conflict via bilevel optimization.*
